In [14]:
# Cell 2: Import libraries
import pandas as pd
import numpy as np
import tensorflow as tf
import autokeras as ak
from sklearn.model_selection import train_test_split
from ast import literal_eval

# Load the CSV into a DataFrame (adjust the file path if necessary)
df = pd.read_csv('~/cerv_spine.csv')

# Display the first few rows to verify the data
df.head()


,url,embedding,study_id,patient_overall,C1,C2,C3,C4,C5,C6,C7
0,https://healthcare.googleapis.com/v1/projects/...,"[1.277871012687683, 0.7918689846992493, -0.400...",1.2.826.0.1.3680043.8744,0,0,0,0,0,0,0,0
1,https://healthcare.googleapis.com/v1/projects/...,"[0.2579282820224762, 0.4537275433540344, -1.02...",1.2.826.0.1.3680043.3234,1,0,0,0,0,0,0,1
2,https://healthcare.googleapis.com/v1/projects/...,"[1.227187275886536, 0.7643394470214844, -0.505...",1.2.826.0.1.3680043.16747,1,0,1,0,0,0,0,0
3,https://healthcare.googleapis.com/v1/projects/...,"[0.06574902683496475, 1.006805777549744, -1.80...",1.2.826.0.1.3680043.25867,1,1,0,0,0,0,0,0
4,https://healthcare.googleapis.com/v1/projects/...,"[0.4398472905158997, 0.5718237161636353, -0.67...",1.2.826.0.1.3680043.26869,0,0,0,0,0,0,0,0


In [6]:
# Convert the "embedding" column from string representation to an actual Python list.
df['embedding'] = df['embedding'].apply(ast.literal_eval)

# Stack the embedding lists into a single 2D NumPy array.
X = np.stack(df['embedding'].values)

# Extract the target variable.
y = df['patient_overall'].values

# Print the shape to verify it matches (number_of_samples, 1408)
print("Input shape:", X.shape)


Input shape: (559, 1408)


In [7]:
# Split the data: 80% training and 20% testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Training samples: 447
Testing samples: 112


In [12]:
# Cell 4: Create AutoKeras model
# Use AutoModel with custom inputs instead of predefined classifier
input_node = ak.Input(shape=(1408,))
output_node = ak.DenseBlock()(input_node)
output_node = ak.ClassificationHead(num_classes=2)(output_node)

# Initialize the AutoModel
model = ak.AutoModel(
    inputs=input_node,
    outputs=output_node,
    max_trials=20,
    overwrite=True,
    project_name='spine_classifier'
)


In [15]:
# Cell 5: Train the model
model.fit(
    X_train, 
    y_train,
    epochs=100,
    validation_split=0.2,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5)]
)


Trial 20 Complete [00h 00m 03s]
val_loss: 0.6384841799736023

Best val_loss So Far: 0.6239274740219116
Total elapsed time: 00h 01m 04s
Epoch 1/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.4902 - loss: 0.9934
Epoch 2/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6041 - loss: 0.6844 
Epoch 3/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6164 - loss: 0.6666 
Epoch 4/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6213 - loss: 0.6401 
Epoch 5/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5994 - loss: 0.6860 
Epoch 6/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6280 - loss: 0.6421 
Epoch 7/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6228 - loss: 0.6420 
Epoch 8/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6247 - loss: 0.6573 
Epoch 9/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6275 - loss: 0.6520 
Epoch 10/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6350 - loss: 0.6291 
Epoc

In [19]:
# Cell 6: Evaluate and export model
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Evaluate on test data
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

# Export the best model
best_model = model.export_model()
best_model.save('spinal_classifier_model.keras')

# Make predictions with the best model
y_pred = model.predict(X_test)
print(f"Shape of predictions: {y_pred.shape}")

auc = roc_auc_score(y_test, y_pred) if y_pred.shape[1] == 1 else roc_auc_score(tf.keras.utils.to_categorical(y_test), y_pred)
print(f"ROC-AUC Score: {auc:.4f}")



4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.6204 - loss: 0.6800 
Test accuracy: 0.6161
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Shape of predictions: (112, 1)
ROC-AUC Score: 0.6197
